In [2]:
import pickle

# Load Geography Encoder
with open('models\ohe.pkl', 'rb') as file:
    ohe_geo = pickle.load(file)

# Load Gender Encoder
with open('models\label_encoder_gender.pkl', 'rb') as file:
    le_gender = pickle.load(file)

# Load Scaler
with open('models\scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [5]:
import pandas as pd

new_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
} 

input_df = pd.DataFrame([new_data])

# Encode Geography
geo_encoded = ohe_geo.transform(input_df[["Geography"]])
geo_df = pd.DataFrame(
    geo_encoded,
    columns=ohe_geo.get_feature_names_out(["Geography"])
)

input_df = input_df.drop("Geography", axis=1)
input_df = pd.concat([input_df, geo_df], axis=1)

# Encode Gender
input_df["Gender"] = le_gender.transform(input_df["Gender"])

# input_df = input_df[X.columns]
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [6]:
num_cols = ["CreditScore","Age","Tenure","Balance","NumOfProducts","EstimatedSalary"]
input_df[num_cols] = scaler.transform(input_df[num_cols])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,-0.535985,1,0.104794,-0.695393,-0.257811,0.808436,1,1,-0.876832,1.0,0.0,0.0


In [9]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

input_tensor = torch.tensor(input_df.values, dtype=torch.float32).to(device)
input_tensor

tensor([[-0.5360,  1.0000,  0.1048, -0.6954, -0.2578,  0.8084,  1.0000,  1.0000,
         -0.8768,  1.0000,  0.0000,  0.0000]])

In [14]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(12, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

model.load_state_dict(torch.load(r"models\best_model.pt", map_location=device))
model.to(device)
model.eval()

input_tensor = input_tensor.to(device)

with torch.no_grad():
    output = model(input_tensor)
    probability = torch.sigmoid(output)
    prediction = (probability > 0.5).float()

print("Probability:", probability.item())
print("Prediction:", int(prediction.item()))

Probability: 0.03179449215531349
Prediction: 0
